In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## READ THE UPLOADED FILE

In [0]:
df = (
    spark.read                             # read is an action-1
        .format('csv')
        .option('header',True)             # header in the first line
        .option('inferSchema',True)        # predict the best schema for our dataframe in CSV format. Action-2
        .load('/Workspace/ApacheSparkGuide/MegaMart.csv')
) 

In [0]:
display(df)

order_id,user_id,order_date,product_id,product_category,product_name,quantity,price_per_unit,payment_method,order_status
1001,U188,2025-04-20,P940,Fashion,Sneakers,2,58.53,PayPal,Cancelled
1002,U062,2025-04-16,P794,Fashion,T-Shirt,3,83.76,UPI,Returned
1003,U058,2025-04-18,P326,Fashion,Sunglasses,2,78.85,PayPal,Processing
1004,U011,2025-04-10,P574,Fashion,Sunglasses,5,46.49,PayPal,Delivered
1005,U003,2025-04-19,P988,Home Decor,Photo Frame,2,78.61,PayPal,Returned
1006,U017,2025-04-15,P328,Kitchen,Knife Set,4,53.51,Credit Card,Returned
1007,U129,2025-04-23,P786,Home Decor,Wall Clock,5,12.71,Credit Card,Returned
1008,U102,2025-04-15,P101,Home Decor,Photo Frame,1,46.6,Debit Card,Cancelled
1009,U040,2025-04-04,P610,Kitchen,Toaster,4,35.87,Credit Card,Processing
1010,U186,2025-04-29,P354,Kitchen,Microwave,1,30.95,Credit Card,Processing


In [0]:
## TRANSFORMATIONS

# Narrow Transformation
df = (
    df
    .filter(col('product_name') == 'Sneakers')
    .select('order_id', 'product_name')
)

# Wide Transformation
df = df.groupBy('product_name').agg(count(col('order_id')))

In [0]:
display(df)

product_name,count(order_id)
Sneakers,41


In [0]:
df.explain('extended')

== Parsed Logical Plan ==
'Aggregate ['product_name], ['product_name, unresolvedalias('count('order_id))]
+- 'Project ['order_id, 'product_name]
   +- 'Filter '`==`('product_name, Sneakers)
      +- Relation [order_id#11295,user_id#11296,order_date#11297,product_id#11298,product_category#11299,product_name#11300,quantity#11301,price_per_unit#11302,payment_method#11303,order_status#11304] csv

== Analyzed Logical Plan ==
product_name: string, count(order_id): bigint
Aggregate [product_name#11300], [product_name#11300, count(order_id#11295) AS count(order_id)#11337L]
+- Project [order_id#11295, product_name#11300]
   +- Filter (product_name#11300 = Sneakers)
      +- Relation [order_id#11295,user_id#11296,order_date#11297,product_id#11298,product_category#11299,product_name#11300,quantity#11301,price_per_unit#11302,payment_method#11303,order_status#11304] csv

== Optimized Logical Plan ==
Aggregate [product_name#11300], [product_name#11300, count(order_id#11295) AS count(order_id)#11337L